# PVSG evaluation: where does the pipeline go wrong?

Compares the pipeline's output (`pvsg_run.ipynb`) with PVSG's human labels, **stage by stage**. No GPU needed.

1. **Objects (SAM2, stages 1–3):** pair each of our tracks with a human-labelled object by mask overlap over the whole video (**vIoU ≥ 0.5**, PVSG's threshold). Human objects with no partner were **missed**; our tracks with no partner are **extra** (or not labelled by PVSG: its labels are not complete).
2. **Names (DAM + LLM, stages 4–5, mapped to PVSG's classes in stage 7):** for each matched pair, is our class the human class?
3. **Relations (stage 6):** for each human relation, why it was or wasn't found:

| outcome | meaning | stage at fault |
|---|---|---|
| correct | same pair, same predicate, time overlap (tIoU) ≥ 0.5 | – |
| wrong time | same pair and predicate, tIoU < 0.5 | 6 (1 frame/s) |
| wrong predicate | the pair is related, but with another predicate | 6 (or the stage-7 mapping) |
| reversed | only the other direction (object → subject) was given | 6 |
| pair not related | both objects found, no relation between them | 6 |
| object missed | subject or object was never found | 1–3 |

In [ ]:
import os, sys, json, glob
import numpy as np, cv2
from PIL import Image
from scipy.optimize import linear_sum_assignment
REPO = "/kaggle/working/svg2-implementation"
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.path.join(os.getcwd(), "third_party/SVG2/pipeline")); sys.path.insert(0, os.path.join(os.getcwd(), "tools"))
import svg2_pipeline as P
import pvsg_data
import matplotlib.pyplot as plt
PVSG_ROOT = "/kaggle/tmp/pvsg" if os.path.isdir("/kaggle/tmp") else "/tmp/pvsg"
RUNS = "/kaggle/working/pvsg_runs"
OUT, DATA = "notes/pvsg", "data/pvsg"
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
anno = pvsg_data.load_anno(PVSG_ROOT)
gt_by_id = {d["video_id"]: d for d in anno["data"]}
VIDEOS = sorted(v for v in os.listdir(RUNS) if os.path.exists(os.path.join(RUNS, v, "pvsg_format.json")))
print("finished videos:", VIDEOS)
PERSON = {"adult", "child", "baby", "person", "man", "woman", "boy", "girl"}

## 1. Objects: match our tracks to the human objects (vIoU over the whole video)

vIoU(ours, human) = (pixels both masks share, summed over all frames) / (pixels in either mask, summed over all frames). Pairs are chosen one-to-one (Hungarian), and kept if vIoU ≥ 0.5.

In [ ]:
def match_objects(v):
    gt = gt_by_id[v]
    source = pvsg_data.video_source(anno, v)
    pngs = sorted(glob.glob(os.path.join(PVSG_ROOT, source, "masks", v, "*.png")))
    tracks = P.read_json(os.path.join(RUNS, v, "stage3_tracks_clean.json"))["objects"]
    gt_ids = [o["object_id"] for o in gt["objects"]]
    size = max(gt_ids + [0]) + 1
    inter = np.zeros((len(tracks), size)); area_ours = np.zeros(len(tracks)); area_gt = np.zeros(size)
    for t, png in enumerate(pngs):
        g = np.array(Image.open(png)).astype(np.int64)
        g[g >= size] = 0
        area_gt += np.bincount(g.ravel(), minlength=size)
        for i, o in enumerate(tracks):
            r = o["masks"][t] if t < len(o["masks"]) else None
            if not r:
                continue
            m = P.decode_rle(r).astype(bool)
            if m.shape != g.shape:
                m = cv2.resize(m.astype(np.uint8), (g.shape[1], g.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
            area_ours[i] += m.sum()
            inter[i] += np.bincount(g[m], minlength=size)
    I = inter[:, gt_ids]
    viou = I / np.maximum(area_ours[:, None] + area_gt[gt_ids][None, :] - I, 1)
    rows, cols = linear_sum_assignment(-viou) if len(tracks) and gt_ids else ([], [])
    pairs = {gt_ids[c]: (tracks[r]["object_id"], float(viou[r, c])) for r, c in zip(rows, cols) if viou[r, c] >= 0.5}
    best = {gid: float(viou[:, j].max()) if len(tracks) else 0.0 for j, gid in enumerate(gt_ids)}
    return pairs, best, [o["object_id"] for o in tracks], len(pngs)

M = {}
for v in VIDEOS:
    M[v] = match_objects(v)
    pairs, best, ours, n = M[v]
    print(f"{v}: {len(pairs)}/{len(best)} human objects found (vIoU ≥ 0.5) by {len(ours)} tracks over {n} frames")

## 2. Objects and names

In [ ]:
obj_rows, name_errors, stuff_as_person = [], [], []
for v in VIDEOS:
    gt = gt_by_id[v]; pairs, best, ours, _ = M[v]
    pred = {o["object_id"]: o for o in json.load(open(os.path.join(RUNS, v, "pvsg_format.json")))["objects"]}
    row = {"video": v, "human things": 0, "things found": 0, "human stuff": 0, "stuff found": 0,
           "our tracks": len(ours), "extra tracks": len(ours) - len(pairs), "named right": 0, "matched & described": 0}
    for o in gt["objects"]:
        kind = "things" if o["is_thing"] else "stuff"
        row[f"human {kind}"] += 1
        if o["object_id"] not in pairs:
            continue
        row[f"{kind} found"] += 1
        oid, vi = pairs[o["object_id"]]
        p = pred.get(oid)
        if p is None:
            continue
        row["matched & described"] += 1
        if p["category"] == o["category"]:
            row["named right"] += 1
        else:
            name_errors.append({"video": v, "gt_id": o["object_id"], "human": o["category"], "is_thing": o["is_thing"],
                                "our_id": oid, "our_name": p["name"], "our_class": p["category"], "viou": round(vi, 2)})
            if not o["is_thing"] and p["category"] in PERSON:
                stuff_as_person.append(name_errors[-1])
    obj_rows.append(row)

cols = list(obj_rows[0]) if obj_rows else []
print(" | ".join(cols))
for r in obj_rows:
    print(" | ".join(str(r[c]) for c in cols))
tot = {c: sum(r[c] for r in obj_rows) for c in cols[1:]}
print("\nTOTAL:", tot)
if tot:
    print(f"things found {tot['things found']}/{tot['human things']}, stuff found {tot['stuff found']}/{tot['human stuff']}, "
          f"named right {tot['named right']}/{tot['matched & described']}")
print(f"\nhuman STUFF (floor, wall, …) that we named as a person: {len(stuff_as_person)}")
for e in stuff_as_person:
    print("  ", e)
print("\nall naming errors (human class → our open name → our mapped class):")
for e in name_errors:
    print(f"  {e['video']} gt {e['gt_id']:>3} {e['human']:>14} → {e['our_name']!r:30} → {e['our_class']}  (vIoU {e['viou']})")

## 3. Relations: for every human relation, what happened

In [ ]:
def frames_of(spans):
    s = set()
    for a, b in spans:
        s.update(range(int(a), int(b) + 1))
    return s

rel_rows, examples = [], []
OUTCOMES = ["correct", "wrong time", "wrong predicate", "reversed", "pair not related", "object missed"]
for v in VIDEOS:
    gt = gt_by_id[v]; pairs, _, _, _ = M[v]
    ours = json.load(open(os.path.join(RUNS, v, "pvsg_format.json")))
    our_of = {g: o for g, (o, _) in pairs.items()}
    by_pair = {}
    for s, o, p, spans in ours["relations"]:
        by_pair.setdefault((s, o), {}).setdefault(p, set()).update(frames_of(spans))
    count = dict.fromkeys(OUTCOMES, 0)
    for s, o, p, spans in gt["relations"]:
        if s not in our_of or o not in our_of:
            out = "object missed"
        else:
            a, b = our_of[s], our_of[o]
            cand = by_pair.get((a, b), {})
            if not cand:
                out = "reversed" if by_pair.get((b, a)) else "pair not related"
            elif p not in cand:
                out = "wrong predicate"
                examples.append((v, s, o, p, sorted(cand)))
            else:
                g, f = frames_of(spans), cand[p]
                out = "correct" if len(g & f) / max(len(g | f), 1) >= 0.5 else "wrong time"
        count[out] += 1
    n_ours = len(ours["relations"])
    matched = {o for o, _ in pairs.values()}
    checkable = [r for r in ours["relations"] if r[0] in matched and r[1] in matched]
    inv = {o: g for g, (o, _) in pairs.items()}
    gt_set = {(s, o, p) for s, o, p, _ in gt["relations"]}
    right_pair_pred = sum((inv[r[0]], inv[r[1]], r[2]) in gt_set for r in checkable)
    none_mapped = sum(r["pvsg_predicate"] == "none" for r in ours["relations_open"])
    rel_rows.append({"video": v, "human relations": len(gt["relations"]), **count, "our relations": n_ours,
                     "ours between found objects": len(checkable), "of those also in PVSG (any time)": right_pair_pred,
                     "our predicates with no PVSG match": none_mapped})

cols = list(rel_rows[0]) if rel_rows else []
for r in rel_rows:
    print(r)
tot_rel = {c: sum(r[c] for r in rel_rows) for c in cols[1:]}
print("\nTOTAL:", tot_rel)
if tot_rel and tot_rel["human relations"]:
    n = tot_rel["human relations"]
    print("\nWhat happened to the human relations:")
    for k in OUTCOMES:
        print(f"  {k:18} {tot_rel[k]:4d}  ({tot_rel[k] / n:.0%})")
    print(f"\nrecall (pair + predicate + time) = {tot_rel['correct'] / n:.1%};  ignoring time = {(tot_rel['correct'] + tot_rel['wrong time']) / n:.1%}")
print("\nwrong-predicate examples (human predicate → what we said for that pair):")
for e in examples[:15]:
    print("  ", e)

## 4. What the naming errors look like
Human mask in red; title = human class vs our name. Up to 6 errors, stuff first.

In [ ]:
shown = sorted(name_errors, key=lambda e: e["is_thing"])[:6]
if shown:
    fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 4), squeeze=False)
    for ax, e in zip(axes[0], shown):
        source = pvsg_data.video_source(anno, e["video"])
        pngs = sorted(glob.glob(os.path.join(PVSG_ROOT, source, "masks", e["video"], "*.png")))
        vis = [t for t, p in enumerate(pngs) if (np.array(Image.open(p)) == e["gt_id"]).any()]
        t = vis[len(vis) // 2]
        cap = cv2.VideoCapture(os.path.join(PVSG_ROOT, source, "videos", e["video"] + ".mp4"))
        cap.set(cv2.CAP_PROP_POS_FRAMES, t); ok, fr = cap.read(); cap.release()
        im = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB).astype(float) / 255
        m = np.array(Image.open(pngs[t])) == e["gt_id"]
        if m.shape != im.shape[:2]:
            m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
        im[m] = 0.5 * im[m] + 0.5 * np.array([1, 0, 0])
        ax.imshow(im); ax.axis("off")
        ax.set_title(f"human: {e['human']}\nours: {e['our_name'][:28]!r}", fontsize=9)
    fig.tight_layout(); fig.savefig(f"{OUT}/naming_errors.png", dpi=70); plt.show()
else:
    print("no naming errors among matched objects")

## 5. Save (small files only) and push

In [ ]:
import shutil
for v in VIDEOS:
    os.makedirs(os.path.join(DATA, v), exist_ok=True)
    for f in ("pvsg_format.json", "stage4_descriptions.json", "status.json"):
        if os.path.exists(os.path.join(RUNS, v, f)):
            shutil.copy(os.path.join(RUNS, v, f), os.path.join(DATA, v, f))
summary = {"videos": VIDEOS, "objects": obj_rows, "objects_total": tot, "naming_errors": name_errors,
           "stuff_named_as_person": stuff_as_person, "relations": rel_rows, "relations_total": tot_rel}
json.dump(summary, open(f"{OUT}/eval_summary.json", "w"), indent=1)
L = ["# PVSG, Option A: pipeline vs human labels", "", f"Videos: {', '.join(VIDEOS)}", "",
     "## Objects", "", "| " + " | ".join(obj_rows[0]) + " |", "|" + "---|" * len(obj_rows[0])]
L += ["| " + " | ".join(str(x) for x in r.values()) + " |" for r in obj_rows]
L += ["", f"Human stuff named as a person: {len(stuff_as_person)}", "", "## Naming errors", ""]
L += [f"- {e['video']}: human `{e['human']}` → ours `{e['our_name']}` (`{e['our_class']}`)" for e in name_errors]
L += ["", "## Relations", "", "| " + " | ".join(rel_rows[0]) + " |", "|" + "---|" * len(rel_rows[0])]
L += ["| " + " | ".join(str(x) for x in r.values()) + " |" for r in rel_rows]
L += ["", "![naming errors](naming_errors.png)"]
open(f"{OUT}/README.md", "w").write("\n".join(L) + "\n")
print("saved", OUT, "and", DATA)
print(f"\nthen:  git add -f {OUT} {DATA} notebooks/pvsg_eval.ipynb && git commit -m 'pvsg eval' && git push")